&nbsp;

## No Proximal CRE transcripts analysis

- public transcriptome (bulk RNA-sequencing data) で選抜した transcripts と、[`fanta.bio`](https://fanta.bio/) を対応付けたとき、TSS ± 500 bp に CRE がない transcript がどのような状態なのかを調べるための notebook
- 粕川先生から、gene level で 10kb までの距離では紐づく gene - cre 対応データをいただいた
- これを元に、reference chromosome 以外の transcript, 10kb 以内に紐づかない transcript (かつ gene level でも紐づかない)、10kb 以内であれば、紐づく transcript の 3つにわける

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt

from IPython.display import display


pl.Config.set_fmt_str_lengths(100)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## `fanta.bio` (ver.1.2.0) gene level CRE association

- 粕川先生から、gene - CRE level relationship file をいただいた
- これは、ver.1.2.0 に該当する
- 10 kb までの対応

In [2]:
# https://data.fanta.bio/cre/v1.2.1/00README_v1.2.1.txt
fantabio_bed = pl.read_csv(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    has_header=False,
    new_columns=[
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "cre_id",
        "score",
        "strand",
        "thickStart",
        "thickEnd",
        "itemRgb",
        "cre_name",
        "pooled_cre_scores"
    ]
).with_columns(
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
).select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class"
)

display(fantabio_bed)

cre_id,cre_name,fantabio_cre_class
str,str,str
"""FCHS_1""","""cp1@DDX11L1""","""PLA"""
"""FCHS_2""","""cp1@WASH7P""","""ELA"""
"""FCHS_447416""","""cp1@DDX11L17""","""ELA"""
"""FCHS_447417""","""FCHS_447417""","""ELA"""
"""FCHS_3""","""cp2@LOC101928626""","""ELA"""
…,…,…
"""FCHS_447311""","""FCHS_447311""","""ELA"""
"""FCHS_447312""","""FCHS_447312""","""ELA"""
"""FCHS_447313""","""FCHS_447313""","""ELA"""


In [3]:
human_gene_cre_association = pl.read_csv(
    "../../Data/Database/fantabio/gene_neighbor/human_gene_cre_associate.csv.gz",
    separator=",",
    n_threads=16
).sort("distance").join(
    fantabio_bed,
    on="cre_id",
    how="inner"
)

display(human_gene_cre_association)

gene_id,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,str,i64,str,str,str,str
"""ENSG00000310526.1""","""FCHS_2""",0,"""human""","""hg38""","""cp1@WASH7P""","""ELA"""
"""ENSG00000243485.6""","""FCHS_2""",0,"""human""","""hg38""","""cp1@WASH7P""","""ELA"""
"""ENSG00000310528.1""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
"""ENSG00000308415.1""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
"""ENSG00000241860.8""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
…,…,…,…,…,…,…
"""ENSG00000298375.1""","""FCHS_253394""",10000,"""human""","""hg38""","""FCHS_253394""","""ELA"""
"""ENSG00000232687.1""","""FCHS_533966""",10000,"""human""","""hg38""","""FCHS_533966""","""ELA"""
"""ENSG00000302188.1""","""FCHS_413906""",10000,"""human""","""hg38""","""FCHS_413906""","""ELA"""


In [5]:
# gene level で CRE と紐づいている gene id を取得
cre_gene_ids = human_gene_cre_association.select(
    pl.col("gene_id").unique().implode()
).to_series()

&nbsp;

&nbsp;

&nbsp;

## GENCODE release 49 GTF file

- `fanta.bio` の CRE に対応しなかった transcript が おそらく haplotype 由来の transcript が含まれている可能性があるため、GTF ファイルの情報から取得する

In [6]:
# For more information, see https://www.gencodegenes.org/pages/biotypes.html
gencode_gtf = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf",
    separator='\t',
    has_header=False,
    comment_prefix='#',
    new_columns=['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attributes'],
    n_threads=16
).filter(
    pl.col('feature') == 'transcript'
).with_columns(
    # Unique keys: extract the first (and only) quoted value
    pl.col("attributes").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_id"),
    pl.col("attributes").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id"),
    pl.col("attributes").str.extract(r'transcript_type "([^"]+)"', 1).alias("transcript_type"),
    # tag can appear multiple times; extract_all returns full matches like tag "basic"
    pl.col("attributes")
        .str.extract_all(r'tag "[^"]+"')
        .list.eval(pl.element().str.extract(r'"([^"]+)"', 1))
        .alias("tags")
).with_columns(
    pl.col("tags").list.n_unique().alias("number_of_tags")
).unique(
    subset=["transcript_id", "transcript_type", "gene_id", "tags", "number_of_tags", "strand"]
).select(
    pl.col("transcript_id"),
    pl.col("seqname"),
    pl.col("transcript_type"),
    pl.col("gene_id"),
    pl.col("tags"),
    pl.col("number_of_tags"),
    pl.col("strand"),
)

display(gencode_gtf)

transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand
str,str,str,str,list[str],u32,str
"""ENST00000526827.1""","""chr11""","""protein_coding""","""ENSG00000149179.15""","[""mRNA_start_NF"", ""cds_start_NF""]",2,"""+"""
"""ENST00000725484.1""","""chr8""","""lncRNA""","""ENSG00000253665.3""","[""TAGENE""]",1,"""+"""
"""ENST00000814997.1""","""chr6""","""lncRNA""","""ENSG00000234263.3""","[""TAGENE""]",1,"""+"""
"""ENST00000378072.5""","""chr20""","""protein_coding""","""ENSG00000101251.13""","[""basic"", ""GENCODE_Primary"", ""CCDS""]",3,"""-"""
"""ENST00000459304.1""","""chr20""","""snRNA""","""ENSG00000238777.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-"""
…,…,…,…,…,…,…
"""ENST00000920877.1""","""chr7""","""protein_coding""","""ENSG00000164885.14""","[""basic"", ""TAGENE""]",2,"""-"""
"""ENST00000733781.1""","""chr22""","""lncRNA""","""ENSG00000230319.2""","[""basic"", ""TAGENE""]",2,"""+"""
"""ENST00000780542.1""","""chr12""","""lncRNA""","""ENSG00000256020.4""","[""basic"", ""TAGENE""]",2,"""-"""


&nbsp;

&nbsp;

&nbsp;

## up5p no proximal CRE

In [7]:
up5p_no_proximal_cre_transcripts = pl.read_parquet(
    "../../out/fantabio/no_proximal_cre_transcirpt_list/up5p_transcript_no_cre_v261001.parquet"
)

print(f"up5p no proximal cre - transcripts: {up5p_no_proximal_cre_transcripts.select('transcript_id').n_unique()}")
display(up5p_no_proximal_cre_transcripts)

up5p no proximal cre - transcripts: 4658


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000391555.1""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000212866.6""","[""basic""]","""+""",1,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000738043.1""",312,9,85,303,195,4,52,191,null,null,"""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1
"""ENST00000400040.1""",309,12,85,297,202,5,44,197,"""HSPA1A""","""HGNC:5232""","""protein_coding""","""ENSG00000215328.6""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000487173.5""",121,77,208,44,64,52,135,12,"""PLAA""","""HGNC:9043""","""protein_coding""","""ENSG00000137055.16""","[""mRNA_start_NF"", ""cds_start_NF""]","""-""",2,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000797768.1""",92,48,266,44,40,29,182,11,null,null,"""lncRNA""","""ENSG00000303875.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A4B8C9""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1
"""ENST00000460240.5""",93,49,264,44,37,29,185,8,"""BRD2""","""HGNC:1103""","""retained_intron""","""ENSG00000215077.10""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0


In [8]:
# gencode_gtf と up5p_no_proximal_cre_transcripts を join
up5p_no_proximal_cre_transcripts_join = (
    gencode_gtf.join(
        up5p_no_proximal_cre_transcripts,
        on=["transcript_id", "transcript_type", "gene_id", "tags", "number_of_tags", "strand"],
        how="inner"
    )
).sort(
    "HN2",
    descending=True
)

pl.Config.set_tbl_rows(114)
print(up5p_no_proximal_cre_transcripts_join.select("seqname").sort("seqname").unique())

pl.Config.set_tbl_rows(4)
display(up5p_no_proximal_cre_transcripts_join)

shape: (114, 1)
┌────────────┐
│ seqname    │
│ ---        │
│ str        │
╞════════════╡
│ GL000195.1 │
│ GL000205.2 │
│ GL000218.1 │
│ GL000219.1 │
│ GL000220.1 │
│ GL000250.2 │
│ GL000251.2 │
│ GL000252.2 │
│ GL000253.2 │
│ GL000254.2 │
│ GL000255.2 │
│ GL000256.2 │
│ GL000258.2 │
│ GL339449.2 │
│ GL383546.1 │
│ GL383564.2 │
│ GL383581.2 │
│ GL949746.1 │
│ GL949747.2 │
│ GL949752.1 │
│ KI270733.1 │
│ KI270742.1 │
│ KI270744.1 │
│ KI270753.1 │
│ KI270791.1 │
│ KI270792.1 │
│ KI270798.1 │
│ KI270803.1 │
│ KI270809.1 │
│ KI270813.1 │
│ KI270816.1 │
│ KI270821.1 │
│ KI270822.1 │
│ KI270823.1 │
│ KI270830.1 │
│ KI270831.1 │
│ KI270832.1 │
│ KI270846.1 │
│ KI270847.1 │
│ KI270849.1 │
│ KI270850.1 │
│ KI270853.1 │
│ KI270857.1 │
│ KI270860.1 │
│ KI270862.1 │
│ KI270866.1 │
│ KI270871.1 │
│ KI270872.1 │
│ KI270875.1 │
│ KI270877.1 │
│ KI270879.1 │
│ KI270895.1 │
│ KI270897.1 │
│ KI270902.1 │
│ KI270903.1 │
│ KI270908.1 │
│ KI270938.1 │
│ KN196474.1 │
│ KN196479.1 │
│ KN196481.1 │
│ KN53836

transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32
"""ENST00000450744.1""","""GL000250.2""","""protein_coding""","""ENSG00000232804.5""","[""basic""]",1,"""+""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000391555.1""","""GL000255.2""","""protein_coding""","""ENSG00000212866.6""","[""basic""]",1,"""+""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""",null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000576236.1""","""GL383564.2""","""protein_coding""","""ENSG00000261986.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",58,14,334,44,32,8,211,24,"""KRT33A""","""HGNC:6450""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Filter non-reference chromosome (haplotypes, scaffold...)

- ここでは、reference chromosome 以外から由来する transcript を選抜する

In [11]:
up5p_no_proximal_cre_transcripts_join_other_chr_check = up5p_no_proximal_cre_transcripts_join.filter(
    ~pl.col("seqname").str.contains("chr")
).with_columns(
    has_cre_gene=pl.col("gene_id").is_in(cre_gene_ids)
).with_columns(
    pl.lit("other chromosome derived").alias("flag")
)

print(up5p_no_proximal_cre_transcripts_join_other_chr_check.select("transcript_id").n_unique())
display(up5p_no_proximal_cre_transcripts_join_other_chr_check)

293


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,bool,str
"""ENST00000450744.1""","""GL000250.2""","""protein_coding""","""ENSG00000232804.5""","[""basic""]",1,"""+""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,false,"""other chromosome derived"""
"""ENST00000391555.1""","""GL000255.2""","""protein_coding""","""ENSG00000212866.6""","[""basic""]",1,"""+""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""",null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0,false,"""other chromosome derived"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000640902.1""","""KV766196.1""","""retained_intron""","""ENSG00000283883.2""",[],0,"""-""",59,15,332,44,31,12,208,19,"""VPS53""","""HGNC:25608""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,false,"""other chromosome derived"""
"""ENST00000576236.1""","""GL383564.2""","""protein_coding""","""ENSG00000261986.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",58,14,334,44,32,8,211,24,"""KRT33A""","""HGNC:6450""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,false,"""other chromosome derived"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

### Filter reference chromosome

- 下流の解析では、reference chromosome (e.g., chr1, chr2...) 由来のデータを選抜する

In [12]:
up5p_no_proximal_cre_transcripts_join_ref_chr = up5p_no_proximal_cre_transcripts_join.filter(
    pl.col("seqname").str.contains("chr")
)

pl.Config.set_tbl_rows(24)
print(up5p_no_proximal_cre_transcripts_join_ref_chr.select("seqname").sort("seqname").unique())

pl.Config.set_tbl_rows(4)
print(up5p_no_proximal_cre_transcripts_join_ref_chr.select("transcript_id").n_unique())
display(up5p_no_proximal_cre_transcripts_join_ref_chr)

shape: (24, 1)
┌─────────┐
│ seqname │
│ ---     │
│ str     │
╞═════════╡
│ chr1    │
│ chr10   │
│ chr11   │
│ chr12   │
│ chr13   │
│ chr14   │
│ chr15   │
│ chr16   │
│ chr17   │
│ chr18   │
│ chr19   │
│ chr2    │
│ chr20   │
│ chr21   │
│ chr22   │
│ chr3    │
│ chr4    │
│ chr5    │
│ chr6    │
│ chr7    │
│ chr8    │
│ chr9    │
│ chrX    │
│ chrY    │
└─────────┘
4365


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32
"""ENST00000738043.1""","""chr6""","""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""-""",312,9,85,303,195,4,52,191,null,null,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1
"""ENST00000626866.2""","""chr13""","""retained_intron""","""ENSG00000120694.21""",[],0,"""-""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000695014.1""","""chr3""","""nonsense_mediated_decay""","""ENSG00000173230.18""","[""mRNA_start_NF"", ""cds_start_NF"", … ""RNA_Seq_supported_only""]",5,"""-""",65,21,320,44,40,14,197,26,"""GOLGB1""","""HGNC:4429""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""alternative 3' UTR"", ""inferred transcript model"", ""RNA-Seq supported only""]",3
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## No Proximal CRE transcirpts

- TSS ± 500 bp にない transcripts が、遺伝子レベルではどの範囲に CRE があるのか調査する

In [13]:
up5p_no_proximal_cre_transcripts_gene_cre_distance = up5p_no_proximal_cre_transcripts_join_ref_chr.join(
    human_gene_cre_association,
    on="gene_id",
    how="left"
).sort(
    "distance",
    descending=True
)


pl.Config.set_tbl_rows(8)
print(f"up5p_no_proximal_cre_transcripts_gene_cre_distance: {up5p_no_proximal_cre_transcripts_gene_cre_distance.select('transcript_id').n_unique()}")
display(up5p_no_proximal_cre_transcripts_gene_cre_distance)

up5p_no_proximal_cre_transcripts_gene_cre_distance: 4365


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str
"""ENST00000622588.2""","""chr9""","""protein_coding""","""ENSG00000273514.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]",6,"""-""",241,22,143,219,152,10,89,142,"""FOXD4L6""","""HGNC:31986""",null,null,null,"""havana_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null
"""ENST00000362482.1""","""chr1""","""rRNA""","""ENSG00000199352.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-""",249,40,117,209,176,20,55,156,"""RNA5S1""","""HGNC:34362""",null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000624004.1""","""chr3""","""TEC""","""ENSG00000279311.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",216,15,175,201,141,7,103,134,null,null,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000601944.1""","""chrX""","""processed_pseudogene""","""ENSG00000273041.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]",4,"""+""",220,20,166,200,139,8,104,131,"""ATF4P2""","""HGNC:34025""",null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_247113""",0,"""human""","""hg38""","""cp10@GIGYF2""","""ELA"""
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_247114""",0,"""human""","""hg38""","""cp8@SNORC""","""ELA"""
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_247115""",0,"""human""","""hg38""","""cp6@SNORC""","""ELA"""
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_527977""",0,"""human""","""hg38""","""cp13@SNORC""","""ELA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. 10 kb 範囲内に CRE が紐づかない transcript

In [18]:
no_cre_10kb_transcripts_check = up5p_no_proximal_cre_transcripts_gene_cre_distance.filter(
    pl.col("distance").is_null()
).sort(
    "HN2",
    descending=True
).with_columns(
    has_cre_gene=pl.col("gene_id").is_in(cre_gene_ids) # gene level で CRE と紐づいているかチェック
).with_columns(
    pl.lit("no gene-level CRE within 10kb").alias("flag")
)

print(f"no_cre_10kb_transcripts: {no_cre_10kb_transcripts_check.select('transcript_id').n_unique()}")
display(no_cre_10kb_transcripts_check)

no_cre_10kb_transcripts: 287


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,bool,str
"""ENST00000622588.2""","""chr9""","""protein_coding""","""ENSG00000273514.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]",6,"""-""",241,22,143,219,152,10,89,142,"""FOXD4L6""","""HGNC:31986""",null,null,null,"""havana_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000362482.1""","""chr1""","""rRNA""","""ENSG00000199352.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-""",249,40,117,209,176,20,55,156,"""RNA5S1""","""HGNC:34362""",null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000624004.1""","""chr3""","""TEC""","""ENSG00000279311.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",216,15,175,201,141,7,103,134,null,null,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000601944.1""","""chrX""","""processed_pseudogene""","""ENSG00000273041.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]",4,"""+""",220,20,166,200,139,8,104,131,"""ATF4P2""","""HGNC:34025""",null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000557832.2""","""chr15""","""TEC""","""ENSG00000284540.1""",[],0,"""-""",106,62,238,44,60,29,162,31,null,null,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000488371.1""","""chr12""","""processed_pseudogene""","""ENSG00000240441.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]",4,"""+""",114,70,222,44,73,43,135,30,"""RPL17P38""","""HGNC:36219""",null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000365488.1""","""chr4""","""snRNA""","""ENSG00000202358.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",70,26,310,44,60,21,170,39,"""RNU6-652P""","""HGNC:47615""",null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000751288.1""","""chrX""","""lncRNA""","""ENSG00000297828.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""+""",54,10,342,44,33,4,214,29,null,null,"""URS0002A42863""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. 10kb 以内であれば gene level で CRE が紐づく transcripts

- distance が 0 ~ ± 500 bp のものがあるが、これは、gene level では TSS に overlap していることを示している

In [19]:
cre_10kb_transcripts_check = up5p_no_proximal_cre_transcripts_gene_cre_distance.filter(
    (pl.col("distance").is_not_null())
).sort(
    "HN2",
    descending=True
).with_columns(
    has_cre_gene=pl.col("gene_id").is_in(cre_gene_ids) # gene level で CRE と紐づいているかチェック
).with_columns(
    pl.lit("gene-level CRE within 10kb").alias("flag")
)

print(cre_10kb_transcripts_check.select("transcript_id").n_unique())
display(cre_10kb_transcripts_check)

4078


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,bool,str
"""ENST00000738043.1""","""chr6""","""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""-""",312,9,85,303,195,4,52,191,null,null,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_359456""",7271,"""human""","""hg38""","""cp10@ELOVL5""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000738043.1""","""chr6""","""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""-""",312,9,85,303,195,4,52,191,null,null,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_566003""",5069,"""human""","""hg38""","""FCHS_566003""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000626866.2""","""chr13""","""retained_intron""","""ENSG00000120694.21""",[],0,"""-""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_482633""",4239,"""human""","""hg38""","""FCHS_482633""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000738043.1""","""chr6""","""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""-""",312,9,85,303,195,4,52,191,null,null,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_566000""",2916,"""human""","""hg38""","""FCHS_566000""","""ELA""",true,"""gene-level CRE within 10kb"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_247113""",0,"""human""","""hg38""","""cp10@GIGYF2""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_247114""",0,"""human""","""hg38""","""cp8@SNORC""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_247115""",0,"""human""","""hg38""","""cp6@SNORC""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000677450.1""","""chr2""","""protein_coding""","""ENSG00000204120.17""","[""mRNA_end_NF""]",1,"""+""",80,36,290,44,65,26,160,39,"""GIGYF2""","""HGNC:11960""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_527977""",0,"""human""","""hg38""","""cp13@SNORC""","""ELA""",true,"""gene-level CRE within 10kb"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Concat

- これまで 3つの 種類のデータフレームに分けて調査してきた
- この3つそれぞれの棒グラフで、かつ transcript type ごとの 積み上げ棒グラフにする

In [24]:
all_no_proximal_cre_transcripts = pl.concat(
    [
        cre_10kb_transcripts_check,
        no_cre_10kb_transcripts_check,
        up5p_no_proximal_cre_transcripts_join_other_chr_check
    ],
    how="diagonal_relaxed" # ない列は null で埋める
)

all_no_proximal_cre_transcripts.write_parquet(
    "../../out/fantabio/no_proximal_cre_transcirpt_list/up5p_transcript_no_cre_v261001_validation.parquet",
    compression="zstd",
    compression_level=9
)

display(all_no_proximal_cre_transcripts)

transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,bool,str
"""ENST00000738043.1""","""chr6""","""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""-""",312,9,85,303,195,4,52,191,null,null,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_359456""",7271,"""human""","""hg38""","""cp10@ELOVL5""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000738043.1""","""chr6""","""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""-""",312,9,85,303,195,4,52,191,null,null,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_566003""",5069,"""human""","""hg38""","""FCHS_566003""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000626866.2""","""chr13""","""retained_intron""","""ENSG00000120694.21""",[],0,"""-""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_482633""",4239,"""human""","""hg38""","""FCHS_482633""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000738043.1""","""chr6""","""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""-""",312,9,85,303,195,4,52,191,null,null,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_566000""",2916,"""human""","""hg38""","""FCHS_566000""","""ELA""",true,"""gene-level CRE within 10kb"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000446351.1""","""GL000254.2""","""unprocessed_pseudogene""","""ENSG00000236746.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",72,28,306,44,45,20,186,25,"""HLA-W""","""HGNC:23425""",null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""other chromosome derived"""
"""ENST00000629053.2""","""KI270903.1""","""protein_coding""","""ENSG00000278165.3""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-""",62,18,326,44,55,14,182,41,"""DUSP8""","""HGNC:3074""",null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""other chromosome derived"""
"""ENST00000640902.1""","""KV766196.1""","""retained_intron""","""ENSG00000283883.2""",[],0,"""-""",59,15,332,44,31,12,208,19,"""VPS53""","""HGNC:25608""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""other chromosome derived"""
"""ENST00000576236.1""","""GL383564.2""","""protein_coding""","""ENSG00000261986.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",58,14,334,44,32,8,211,24,"""KRT33A""","""HGNC:6450""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""other chromosome derived"""
